# Phase 3 - Winner calibration, eval-test scoring, and explainability

This notebook loads the winner from `results_phase2.zip`, calibrates a domain-agnostic threshold on labelled `dev_test`, scores unlabeled `eval_test`, and generates **reconstruction-based explainability plots** (mel-bin attribution + PCA of bottleneck features + example error maps).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/Unsupervised_anomalous_sound

import os, json, glob, zipfile, shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
)
from IPython.display import Markdown, display

from model import build_model
from dataset import BearingDataset
from dataset_phase2 import BearingTestDataset

In [ ]:
# ---- paths ----
RESULTS_ZIP = "results_phase2.zip"
EXTRACT_DIR = "/content/results_phase2"
OUT_DIR = "/content/phase3_outputs"
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(RESULTS_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)

winner_path = os.path.join(EXTRACT_DIR, "best_model", "WINNER.json")
with open(winner_path) as f:
    winner = json.load(f)

key = winner["key"]
arch = winner["arch"]
ckpts = sorted(glob.glob(os.path.join(EXTRACT_DIR, "best_model", f"{key}_fold*.pt")))
print("Winner:", winner)
print("Checkpoints:", len(ckpts))

In [ ]:
# ---- shared scoring helper ----
device = "cuda" if torch.cuda.is_available() else "cpu"

def score_with_ensemble(loader, arch, ckpt_paths):
    fold_scores = []
    domains_ref, sections_ref, labels_ref, files_ref = None, None, None, None

    for ckpt_path in ckpt_paths:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = build_model(arch).to(device)
        model.load_state_dict(ckpt["model_state"])
        model.eval()

        scores, domains, sections, labels, files = [], [], [], [], []
        with torch.no_grad():
            for batch in loader:
                x = batch[0].to(device, non_blocking=True)
                recon, _ = model(x)
                t = min(recon.shape[-1], x.shape[-1])
                err = ((recon[..., :t] - x[..., :t]) ** 2).mean(dim=[1, 2, 3]).cpu().numpy()
                scores.append(err)

                # For BearingDataset(return_label=True): (x, domain, section, label)
                # For BearingTestDataset:               (x, domain, section)
                dom = list(batch[1])
                sec = [int(s) for s in batch[2]]
                lab = list(batch[3]) if len(batch) > 3 else ["unknown"] * len(dom)
                domains.extend(dom)
                sections.extend(sec)
                labels.extend(lab)

        fold_scores.append(np.concatenate(scores))
        if domains_ref is None:
            domains_ref = np.array(domains)
            sections_ref = np.array(sections)
            labels_ref = np.array(labels)

    ens_scores = np.mean(np.stack(fold_scores, axis=0), axis=0)
    return ens_scores, domains_ref, sections_ref, labels_ref

In [ ]:
# ---- 1) Calibrate threshold on labelled dev_test (domain-agnostic) ----
dev_test_dir = "data/dcase_bearing_dev/bearing/test"
dev_test_cache = "/content/bearing_cache_dev_test"
dev_ds = BearingDataset(dev_test_dir, cache_dir=dev_test_cache, return_label=True, verbose=False)
dev_loader = DataLoader(dev_ds, batch_size=128, shuffle=False, num_workers=0)

dev_scores, dev_domains, dev_sections, dev_labels = score_with_ensemble(dev_loader, arch, ckpts)
y_true = (dev_labels == "anomaly").astype(int)

prec, rec, thr = precision_recall_curve(y_true, dev_scores)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
best_i = int(np.nanargmax(f1))
best_thr = float(thr[best_i])
y_pred = (dev_scores >= best_thr).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
metrics = {
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "roc_auc": float(roc_auc_score(y_true, dev_scores)),
    "average_precision": float(average_precision_score(y_true, dev_scores)),
    "threshold": best_thr,
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
}

display(Markdown(
    f"""
### Domain-agnostic threshold (calibrated on dev_test)
- threshold = `{metrics['threshold']:.6f}`
- precision = `{metrics['precision']:.4f}`
- recall = `{metrics['recall']:.4f}`
- F1 = `{metrics['f1']:.4f}`
- accuracy = `{metrics['accuracy']:.4f}`
- ROC-AUC = `{metrics['roc_auc']:.4f}`
- PR-AUC = `{metrics['average_precision']:.4f}`
- confusion = `TN={tn}, FP={fp}, FN={fn}, TP={tp}`
"""
))

with open(os.path.join(OUT_DIR, "threshold_metrics.json"), "w") as f:
    json.dump({"winner": winner, "domain_agnostic": True, "metrics": metrics}, f, indent=2)

## Explainability (what separates normal vs anomaly)

This section is **post-hoc** explanation of the winner's anomaly score = mean squared reconstruction error (MSE).

### What `adversarial` changed during training

`adversarial` adds a small domain-classification loss on the bottleneck features `z` (source vs target). At **inference**, we still score clips with reconstruction MSE only — the domain head is not used.

So explainability here answers: **which time-frequency regions drive high reconstruction error for anomalies** (and whether that pattern is consistent across sections/domains).

### Plots produced

- `explain_melbin_mean_error.png` — average squared error per mel bin for normals vs anomalies (pooled dev-test).
- `explain_pca_latent.png` — PCA of pooled bottleneck vectors `z` (2D), colored by label/section/domain.
- `explain_examples/` — a few side-by-side panels: input log-mel, reconstruction, squared error map.

In [ ]:
# ---- 1b) Explainability on dev_test (winner ensemble) ----
from sklearn.decomposition import PCA

explain_dir = os.path.join(OUT_DIR, "explain")
os.makedirs(explain_dir, exist_ok=True)

models = []
for ckpt_path in ckpts:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = build_model(arch).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    models.append(m)

# --- A) Mel-bin attribution: mean squared error averaged over time ---
mel_err_sum = {"normal": None, "anomaly": None}
mel_err_cnt = {"normal": 0, "anomaly": 0}

# --- B) Latent PCA features (pooled z) ---
Z_list, y_list, dom_list, sec_list = [], [], [], []

with torch.no_grad():
    for batch in dev_loader:
        x = batch[0].to(device, non_blocking=True)
        dom = list(batch[1])
        sec = [int(s) for s in batch[2]]
        lab = list(batch[3])

        # fold-mean squared error map + fold-mean recon for viz
        err_maps = []
        recons = []
        zs = []
        for m in models:
            recon, z = m(x)
            t = min(recon.shape[-1], x.shape[-1])
            e = (recon[..., :t] - x[..., :t]) ** 2  # (B,1,H,T)
            err_maps.append(e.detach())
            recons.append(recon[..., :t].detach())
            zp = z.mean(dim=[2, 3]).detach().cpu().numpy()
            zs.append(zp)

        err_map = torch.stack(err_maps, dim=0).mean(dim=0)  # (B,1,H,T)
        recon_m = torch.stack(recons, dim=0).mean(dim=0)
        z_ens = np.mean(np.stack(zs, axis=0), axis=0)  # (B, C)

        # mel-bin curves (H,)
        mel_curve = err_map.mean(dim=(1, 3)).squeeze(1).cpu().numpy()  # (B, H)

        for i in range(x.size(0)):
            tag = "anomaly" if lab[i] == "anomaly" else "normal"
            v = mel_curve[i]
            if mel_err_sum[tag] is None:
                mel_err_sum[tag] = v.astype(np.float64)
            else:
                mel_err_sum[tag] += v.astype(np.float64)
            mel_err_cnt[tag] += 1

            Z_list.append(z_ens[i])
            y_list.append(1 if lab[i] == "anomaly" else 0)
            dom_list.append(dom[i])
            sec_list.append(sec[i])

bins = np.arange(mel_err_sum["normal"].shape[0])
plt.figure(figsize=(9, 4.5))
plt.plot(bins, mel_err_sum["normal"] / max(mel_err_cnt["normal"], 1), label="normal (mean sq err / mel bin)")
plt.plot(bins, mel_err_sum["anomaly"] / max(mel_err_cnt["anomaly"], 1), label="anomaly (mean sq err / mel bin)")
plt.plot(
    bins,
    (mel_err_sum["anomaly"] / max(mel_err_cnt["anomaly"], 1))
    - (mel_err_sum["normal"] / max(mel_err_cnt["normal"], 1)),
    label="difference (anomaly - normal)",
    color="k",
    alpha=0.6,
)
plt.xlabel("mel bin")
plt.ylabel("mean squared reconstruction error")
plt.title("Mel-bin attribution (dev-test, domain pooled)")
plt.grid(alpha=0.3)
plt.legend()
p_mel = os.path.join(explain_dir, "explain_melbin_mean_error.png")
plt.tight_layout()
plt.savefig(p_mel, dpi=140)
plt.close()

Z = np.stack(Z_list, axis=0)
y = np.array(y_list)
pca = PCA(n_components=2, random_state=0)
Z2 = pca.fit_transform(Z)

plt.figure(figsize=(6.2, 5.2))
plt.scatter(Z2[y == 0, 0], Z2[y == 0, 1], s=18, alpha=0.55, label="normal")
plt.scatter(Z2[y == 1, 0], Z2[y == 1, 1], s=18, alpha=0.55, label="anomaly")
plt.title("PCA of pooled bottleneck z (ensemble mean across folds)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca = os.path.join(explain_dir, "explain_pca_latent_label.png")
plt.tight_layout()
plt.savefig(p_pca, dpi=140)
plt.close()

sec_arr = np.array(sec_list)
plt.figure(figsize=(6.2, 5.2))
for s in sorted(set(sec_arr.tolist())):
    m = sec_arr == s
    plt.scatter(Z2[m, 0], Z2[m, 1], s=18, alpha=0.55, label=f"sec {s}")
plt.title("PCA of z colored by section")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca_sec = os.path.join(explain_dir, "explain_pca_latent_section.png")
plt.tight_layout()
plt.savefig(p_pca_sec, dpi=140)
plt.close()

dom_arr = np.array(dom_list)
plt.figure(figsize=(6.2, 5.2))
for d in ["source", "target"]:
    m = dom_arr == d
    plt.scatter(Z2[m, 0], Z2[m, 1], s=18, alpha=0.55, label=d)
plt.title("PCA of z colored by domain")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.legend()
p_pca_dom = os.path.join(explain_dir, "explain_pca_latent_domain.png")
plt.tight_layout()
plt.savefig(p_pca_dom, dpi=140)
plt.close()

# --- C) Example panels: pick a few highest-error anomalies + a few lowest-error normals ---

def _example_panel(idx, tag):
    sample = dev_ds.samples[idx]
    x = dev_ds[idx][0].unsqueeze(0).to(device)
    with torch.no_grad():
        err_maps = []
        recons = []
        for m in models:
            recon, _ = m(x)
            t = min(recon.shape[-1], x.shape[-1])
            e = (recon[..., :t] - x[..., :t]) ** 2
            err_maps.append(e)
            recons.append(recon[..., :t])
        err = torch.stack(err_maps, dim=0).mean(dim=0).detach().cpu().numpy()
        err = np.squeeze(err)  # (H, T)
        rec = torch.stack(recons, dim=0).mean(dim=0).detach().cpu().numpy()
        rec = np.squeeze(rec)
    inp = x.detach().cpu().numpy()
    inp = np.squeeze(inp)
    if inp.ndim != 2:
        raise ValueError(f"expected 2D log-mel for imshow, got shape {inp.shape}")

    vmax = float(np.percentile(err, 99.0))
    fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
    ax[0].imshow(inp, aspect="auto", origin="lower")
    ax[0].set_title("input log-mel")
    ax[1].imshow(rec, aspect="auto", origin="lower")
    ax[1].set_title("recon (ens.)")
    im = ax[2].imshow(err, aspect="auto", origin="lower", vmin=0.0, vmax=max(vmax, 1e-8))
    ax[2].set_title("squared error")
    for a in ax:
        a.set_xlabel("time")
        a.set_ylabel("mel")
    fig.suptitle(f"{tag}: {sample['filename']}  (label={sample['label']}, dom={sample['domain']}, sec={sample['section']})")
    fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
    fig.tight_layout()
    out = os.path.join(explain_dir, f"example_{tag}_{idx}.png")
    fig.savefig(out, dpi=160)
    plt.close(fig)
    return out

order = np.argsort(-dev_scores)  # high error first
anom_idx = [i for i in order.tolist() if dev_ds.samples[i]["label"] == "anomaly"][:3]
norm_idx = [i for i in np.argsort(dev_scores).tolist() if dev_ds.samples[i]["label"] == "normal"][:3]

example_paths = []
for i in anom_idx:
    example_paths.append(_example_panel(i, "high_err_anomaly"))
for i in norm_idx:
    example_paths.append(_example_panel(i, "low_err_normal"))

summary_x = {
    "winner": winner,
    "note": "Explainability is based on reconstruction error and bottleneck embeddings; domain classifier is not used at inference.",
    "plots": {
        "melbin": p_mel,
        "pca_label": p_pca,
        "pca_section": p_pca_sec,
        "pca_domain": p_pca_dom,
        "examples": example_paths,
    },
}
with open(os.path.join(explain_dir, "explain_summary.json"), "w") as f:
    json.dump(summary_x, f, indent=2)

display(Markdown("### Explainability plots"))
for path in [p_mel, p_pca, p_pca_sec, p_pca_dom] + example_paths:
    display(Image(filename=path))


In [ ]:
# ---- 2) Score unlabeled eval_test and write per-section CSVs ----
eval_test_dir = "data/dcase_bearing_eval/bearing/test"
eval_test_cache = "/content/bearing_cache_eval_test"
eval_ds = BearingTestDataset(eval_test_dir, cache_dir=eval_test_cache, verbose=False)
eval_loader = DataLoader(eval_ds, batch_size=128, shuffle=False, num_workers=0)

eval_scores, eval_domains, eval_sections, eval_labels = score_with_ensemble(eval_loader, arch, ckpts)
eval_pred = (eval_scores >= best_thr).astype(int)

records = []
for i, s in enumerate(eval_ds.samples):
    records.append({
        "filename": s["filename"],
        "section": int(s["section"]),
        "score": float(eval_scores[i]),
        "decision": int(eval_pred[i]),
    })

sections = sorted(set(r["section"] for r in records))
for sec in sections:
    sec_rows = sorted([r for r in records if r["section"] == sec], key=lambda x: x["filename"])
    score_path = os.path.join(OUT_DIR, f"anomaly_score_section_{sec:02d}.csv")
    decision_path = os.path.join(OUT_DIR, f"decision_result_section_{sec:02d}.csv")
    with open(score_path, "w") as f:
        for r in sec_rows:
            f.write(f"{r['filename']},{r['score']:.10f}\n")
    with open(decision_path, "w") as f:
        for r in sec_rows:
            f.write(f"{r['filename']},{r['decision']}\n")

summary = {
    "winner": winner,
    "threshold": best_thr,
    "n_eval_clips": len(records),
    "sections": sections,
    "mean_score": float(np.mean(eval_scores)),
    "std_score": float(np.std(eval_scores)),
    "predicted_anomaly_rate": float(np.mean(eval_pred)),
}
with open(os.path.join(OUT_DIR, "eval_test_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

display(Markdown(
    f"""
### Eval-test scoring complete
- clips scored: `{len(records)}`
- sections: `{sections}`
- threshold used: `{best_thr:.6f}`
- predicted anomaly rate: `{np.mean(eval_pred):.4f}`
- outputs: `anomaly_score_section_XX.csv`, `decision_result_section_XX.csv`
"""
))

In [ ]:
# ---- 3) zip outputs + copy to Drive + download ----
zip_base = "/content/phase3_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Created:", zip_path)

drive_zip = os.path.join(os.getcwd(), "phase3_outputs.zip")
if os.path.exists(drive_zip):
    os.remove(drive_zip)
shutil.copy2(zip_path, drive_zip)
print("Copied to Drive project folder:", drive_zip)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download skipped:", e)